# Õppetund 01 - Sissejuhatus tehisintellekti agendidesse

Tere tulemast esimesse õppetundi kursusel **AI Agents for Beginners**!

**Tehisintellekti agent** on programm, mis kasutab suurt keelemudelit (LLM) oma mõtlemismootorina ning suudab teha *tegevusi* pärismaailmas — kutsuda API-sid, pärida andmebaase või käivitada koodi — et saavutada kasutaja nimel eesmärk.

Selles märkmikus ehitad oma esimese agendi: **Reisiagendi**, mis soovitab puhkuse sihtkohti. Selle käigus õpid, kuidas:

1. Ühenduda Azure AI Foundry Agent Service’iga, kasutades **Microsoft Agent Frameworki**.
2. Anda agendile **tööriist** — lihtsa Pythoni funktsiooni, mida ta saab kutsuda.
3. Käivitada agent ja uurida selle vastust.
4. Voogesitada agendi vastust token-tokeni haaval.


## Seadistamine

Enne selle märkmiku käivitamist veenduge, et teil on:

1. **Azure AI Foundry projekt** koos juurutatud vestlusmudeliga (nt `gpt-4o-mini`).
2. **Sisselogitud Azure CLI-ga** — käivitage terminalis `az login`.
3. **Seatud vajalikud keskkonnamuutujad:**
   - `AZURE_AI_PROJECT_ENDPOINT` — teie Azure AI Foundry projekti lõpp-punkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — teie juurutatud mudeli nimi.

Allolev lahter installib vajalikud Python paketid.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Oma esimese agendi loomine

Agendil on vaja kahte asja:

- **Juhised**, mis ütlevad talle *kes ta on* ja *kuidas käituda* (süsteemiprompt).
- **Tööriistad** — Python funktsioonid, millel on `@tool` dekoratsioon, mida agent saab kutsuda info saamiseks või toimingute tegemiseks.

Allpool määratleme lihtsa tööriista, mis tagastab nimekirja populaarsetest puhkuse sihtkohtadest. Agent kasutab seda tööriista, kui kasutaja palub reisisoovitusi.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Voogedastuse vastused

Interaktiivsema kogemuse jaoks saate agendi vastuse **voogedastada**. Selle asemel, et oodata kogu vastust, annab agent tekstitükke välja kohe, kui need genereeritakse. See on eriti kasulik vestlusliidestes, kus soovite väljundit reaalajas kuvada.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Kokkuvõte

Selles õppetükis õppisid sa, kuidas:

- **Luua pakkuja**, mis ühendub Azure AI Foundry Agent Service'iga läbi `AzureAIProjectAgentProvider`.
- **Määratleda tööriist** kasutades `@tool` dekoratsiooni, et agent saaks kutsuda sinu Python funktsioone.
- **Käivita agent** kasutajateatega ja kuva selle vastus.
- **Voogesitada vastuseid** reaalaja väljundi jaoks.

Järgmises õppetükis uurime agentide raamistikke põhjalikumalt ning õpime, kuidas anda agentidele võimsamaid tööriistu ja mitmeastmelise mõtlemise võimekust.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:  
See dokument on tõlgitud kasutades tehisintellektil põhinevat tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi püüame tagada täpsust, palun arvestage, et automatiseeritud tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle algkeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate arusaamatuste või valesti mõistmiste eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
